# CS1-EXP1 — Within-Project StratifiedKFold Comparison

## Purpose

This notebook repeats the frozen EXP-1 architecture under the same
**within-project StratifiedKFold manifest** used by
`03_cs1_exp0_stratifiedkfold.ipynb`.

```text
Word TF-IDF + Character TF-IDF
→ train-only TruncatedSVD (256 components)
→ 54 deterministic static features
→ Random Forest (200 trees)
→ training-only OOB F1 threshold selection
→ 5-fold StratifiedKFold OOF evaluation
```

## Controlled comparison

Only the fold assignment changes relative to the grouped EXP-1 experiment.

| Fixed component | Value |
|---|---|
| Dataset / normalization | Same |
| Static-feature cache | Same |
| TF-IDF configuration | Same |
| SVD components | 256 |
| Random Forest | 200 trees, 70% bootstrap, `balanced_subsample` |
| Threshold protocol | Training-only OOB F1 selection |
| Random seed | 42 |
| Changed component | Grouped folds → ordinary StratifiedKFold |

> This is a **secondary within-project evaluation**. Functions from the same
> projects occur in both train and test partitions by design. Do not interpret
> it as unseen-project generalization.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Define paths

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/IntelligentSystemProject/VulnerabilityDetectionData"
)

PROCESSED_DIR = DRIVE_ROOT / "processed"
MANIFEST_DIR = DRIVE_ROOT / "manifests"
OUTPUT_ROOT = DRIVE_ROOT / "outputs"

NORMALIZED_DATA_PATH = (
    PROCESSED_DIR / "rdiversevul_cs1_normalized_v1.parquet"
)

STATIC_FEATURE_PATH = (
    PROCESSED_DIR
    / "static_features"
    / "cs1_static_features_v1.parquet"
)

WITHIN_PROJECT_MANIFEST_PATH = (
    MANIFEST_DIR / "cs1_within_project_stratified_5fold_manifest.parquet"
)

EXP1_OUTPUT_ROOT = OUTPUT_ROOT / "cs1_exp1_rf"
EXP1_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Normalized data:", NORMALIZED_DATA_PATH)
print("Static feature cache:", STATIC_FEATURE_PATH)
print("Within-project manifest:", WITHIN_PROJECT_MANIFEST_PATH)
print("EXP-1 output root:", EXP1_OUTPUT_ROOT)

## 3. Clone or refresh the `prashant` GitHub branch

In [ ]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
BRANCH = "prashant"
REPO_DIR = Path("/content/DiverseVul--IS-Project")

if not REPO_DIR.exists():
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    os.chdir(REPO_DIR)
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}

PROJECT_DIR = REPO_DIR / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.chdir(PROJECT_DIR)

print("Repository:", REPO_DIR)
print("Branch:", BRANCH)
print("Working directory:", Path.cwd())

## 4. Install dependencies

In [ ]:
!pip -q install \
    numpy \
    pandas \
    scipy \
    scikit-learn \
    matplotlib \
    pyarrow \
    joblib

## 5. Import the within-project EXP-1 module

Ensure the repository contains:

```text
src/case_study_1/within_project_manifest.py
src/case_study_1/exp1_rf_within_project.py
```


In [ ]:
import pandas as pd
import numpy as np

import case_study_1.within_project_manifest as within_project_manifest
import case_study_1.exp1_rf_within_project as exp1_wp
import case_study_1.evaluation as evaluation

print("Manifest version:", within_project_manifest.MANIFEST_VERSION)
print("EXP-1 version:", exp1_wp.EXP1_VERSION)

## 6. Load the frozen normalized dataset, static cache, and StratifiedKFold manifest

In [ ]:
required_paths = [
    NORMALIZED_DATA_PATH,
    STATIC_FEATURE_PATH,
    WITHIN_PROJECT_MANIFEST_PATH,
]

for required_path in required_paths:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required artifact not found: {required_path}"
        )

normalized_df = pd.read_parquet(NORMALIZED_DATA_PATH)
static_df = pd.read_parquet(STATIC_FEATURE_PATH)

within_config = within_project_manifest.WithinProjectSplitConfig(
    n_splits=5,
    random_state=42,
    shuffle=True,
    source_id_column="source_row_id",
    label_column="label",
    project_column="project",
)

within_manifest_df = (
    within_project_manifest.load_within_project_manifest(
        WITHIN_PROJECT_MANIFEST_PATH,
        config=within_config,
    )
)

print("Normalized dataset:", normalized_df.shape)
print("Static-feature cache:", static_df.shape)
print("Within-project manifest:", within_manifest_df.shape)

## 7. Integrity checks

In [ ]:
assert len(normalized_df) == 261_667
assert normalized_df["source_row_id"].nunique() == len(normalized_df)
assert set(normalized_df["label"].unique()) == {0, 1}
assert normalized_df["normalized_code"].notna().all()

assert len(static_df) == len(normalized_df)
assert static_df["source_row_id"].nunique() == len(normalized_df)

assert len(within_manifest_df) == len(normalized_df)
assert within_manifest_df["source_row_id"].nunique() == len(normalized_df)
assert set(within_manifest_df["fold"].unique()) == {0, 1, 2, 3, 4}

within_summary_df = (
    within_project_manifest.summarize_within_project_manifest(
        within_manifest_df,
        config=within_config,
    )
)

assert (within_summary_df["test_row_overlap_rate"] > 0.95).all()
assert (within_summary_df["overlap_rate_of_test_projects"] > 0.95).all()

print("✅ Dataset, static cache, and manifest row IDs align.")
print("✅ StratifiedKFold class balance is preserved.")
print("✅ Project overlap is deliberately present in every fold.")
display(
    within_summary_df[
        [
            "fold",
            "test_rows",
            "test_positive_rate",
            "test_unique_projects",
            "train_test_project_overlap",
            "overlap_rate_of_test_projects",
            "test_row_overlap_rate",
        ]
    ].style.format(
        {
            "test_positive_rate": "{:.4%}",
            "overlap_rate_of_test_projects": "{:.2%}",
            "test_row_overlap_rate": "{:.2%}",
        }
    )
)

## 8. Declare the identical EXP-1 model configuration

Do not tune these values after seeing within-project results. They are
identical to the completed grouped EXP-1 configuration except for the manifest.


In [ ]:
exp1_within_config = exp1_wp.Exp1Config(
    experiment_name="cs1_exp1_rf_within_project_stratified_oob",

    # Same outer fold count and seed.
    n_splits=5,
    random_state=42,
    decision_threshold=0.50,

    # Same lexical representation.
    word_ngram_range=(1, 3),
    word_min_df=3,
    word_max_df=0.995,
    word_max_features=50_000,

    char_analyzer="char",
    char_ngram_range=(3, 4),
    char_min_df=8,
    char_max_df=0.995,
    char_max_features=60_000,

    # Same train-only SVD.
    svd_n_components=256,
    svd_algorithm="randomized",
    svd_n_iter=5,
    svd_n_oversamples=10,

    # Same Random Forest settings.
    rf_n_estimators=200,
    rf_criterion="gini",
    rf_max_depth=None,
    rf_min_samples_split=2,
    rf_min_samples_leaf=2,
    rf_max_features="sqrt",
    rf_bootstrap=True,
    rf_max_samples=0.70,
    rf_class_weight="balanced_subsample",
    rf_n_jobs=-1,

    # Same training-only OOB operating-point selection.
    rf_oob_score=True,
    oob_threshold_min=0.005,
    oob_threshold_max=0.250,
    oob_threshold_step=0.005,
    oob_threshold_objective="f1",

    feature_importance_top_n=50,
    verbose=True,
)

print(exp1_within_config)

## 9. Profile Fold 0 only

This checks runtime, OOB threshold selection, and the deliberate project overlap.
It is not the official five-fold result.


In [ ]:
exp1_within_profile = exp1_wp.run_exp1_profile_fold(
    normalized_frame=normalized_df,
    static_features_frame=static_df,
    manifest=within_manifest_df,
    fold_id=0,
    config=exp1_within_config,
)

print("\nFold 0 training metadata:")
display(
    exp1_within_profile["training_metadata"][
        [
            "fold",
            "train_rows",
            "test_rows",
            "train_unique_projects",
            "test_unique_projects",
            "train_test_project_overlap",
            "overlap_rate_of_test_projects",
            "word_tfidf_seconds",
            "char_tfidf_seconds",
            "svd_seconds",
            "model_fit_seconds",
            "total_fold_seconds",
            "selected_oob_threshold",
            "oob_precision",
            "oob_recall",
            "oob_f1",
            "oob_mcc",
            "svd_explained_variance_ratio_sum",
        ]
    ]
)

print("\nHeld-out Fold 0 metrics at training-OOB selected threshold:")
print(
    evaluation.format_metric_report(
        exp1_within_profile["profile_metrics"]
    )
)

print("\nDiagnostic only: Fold 0 metrics at fixed threshold 0.50:")
print(
    evaluation.format_metric_report(
        exp1_within_profile["default_threshold_metrics"]
    )
)

## 10. Official five-fold within-project EXP-1 run

Run this cell only after the Fold 0 profile succeeds.

A fresh folder prevents accidental mixing with the grouped EXP-1 artifacts.


In [ ]:
WITHIN_PROJECT_EXP1_OUTPUT_DIR = (
    EXP1_OUTPUT_ROOT / "official_within_project_stratified_oob_v1"
)

WITHIN_PROJECT_EXP1_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

existing_output_files = list(
    WITHIN_PROJECT_EXP1_OUTPUT_DIR.iterdir()
)

if existing_output_files:
    raise RuntimeError(
        "Official output folder is not empty. "
        "Use a fresh directory or deliberately inspect/remove only incomplete artifacts."
    )

exp1_within_results = exp1_wp.run_exp1(
    normalized_frame=normalized_df,
    static_features_frame=static_df,
    manifest=within_manifest_df,
    config=exp1_within_config,
    output_dir=WITHIN_PROJECT_EXP1_OUTPUT_DIR,
    additional_metadata={
        "run_type": "official_full_5fold_oof_evaluation",
        "evaluation_protocol": "within_project_stratified",
        "protocol_interpretation": (
            "Secondary evaluation: the same projects are present in training "
            "and test partitions by design. This measures within-project "
            "performance and must not be interpreted as unseen-project generalization."
        ),
        "normalized_dataset_path": str(NORMALIZED_DATA_PATH),
        "static_feature_cache_path": str(STATIC_FEATURE_PATH),
        "manifest_path": str(WITHIN_PROJECT_MANIFEST_PATH),
        "model_configuration_control": (
            "Identical to grouped EXP-1 except for the frozen fold assignment."
        ),
        "primary_comparator": (
            "official_svd_static_rf_oob_v2 using StratifiedGroupKFold by project"
        ),
    },
)

print("Standard fixed-threshold diagnostic:")
print(
    evaluation.format_metric_report(
        exp1_within_results["evaluation"]["pooled_metrics"]
    )
)

print("\nOOB-selected operating metrics:")
print(
    exp1_within_results["oob_operating_evaluation"]["pooled_metrics"]
)

## 11. Final comparison-ready outputs

In [ ]:
print("Per-fold OOB-selected operating metrics:")
display(
    exp1_within_results["oob_operating_evaluation"]["fold_metrics"][
        [
            "fold",
            "threshold",
            "n_samples",
            "test_unique_projects",
            "positive_rate",
            "average_precision_pr_auc",
            "precision",
            "recall",
            "f1",
            "mcc",
            "false_positive_rate",
            "predicted_positive_rate",
        ]
    ]
)

print("\nTraining runtime and OOB threshold selection:")
display(
    exp1_within_results["fold_training"][
        [
            "fold",
            "train_rows",
            "test_rows",
            "train_test_project_overlap",
            "overlap_rate_of_test_projects",
            "word_tfidf_seconds",
            "char_tfidf_seconds",
            "svd_seconds",
            "model_fit_seconds",
            "total_fold_seconds",
            "selected_oob_threshold",
            "oob_precision",
            "oob_recall",
            "oob_f1",
            "oob_mcc",
        ]
    ]
)

importance_by_group = (
    exp1_within_results["feature_importances"]
    .groupby(["fold", "feature_group"], as_index=False)
    .agg(total_importance=("importance", "sum"))
)

importance_by_group["importance_share"] = (
    importance_by_group.groupby("fold")["total_importance"]
    .transform(lambda values: values / values.sum())
)

print("\nFeature-group importance:")
display(importance_by_group)

print("\nAverage feature-group importance:")
display(
    importance_by_group
    .groupby("feature_group", as_index=False)
    .agg(
        mean_importance_share=("importance_share", "mean"),
        std_importance_share=("importance_share", "std"),
    )
    .style.format(
        {
            "mean_importance_share": "{:.2%}",
            "std_importance_share": "{:.2%}",
        }
    )
)